## Image pull secrets

If a pod runs an image from a **private registry**, the kubelet on the node needs credentials to pull it. That's what `imagePullSecrets` is for.

**Step 1 — create the credential as a Secret:**

```bash
kubectl create secret docker-registry regcred \
  --docker-server=registry.example.com \
  --docker-username=ci-bot \
  --docker-password='<password>'
```

That makes a `kubernetes.io/dockerconfigjson` Secret whose `data` is a base64 of a `~/.docker/config.json`.

**Step 2 — reference it from the pod spec:**

```yaml
spec:
  imagePullSecrets:
    - name: regcred
  containers:
    - name: app
      image: registry.example.com/team/app:1.2.3
```

The kubelet uses `regcred` to authenticate the pull. Crucially, the Secret is **only** used for image pull — the running container never sees it as an env var or file.

### Cluster-wide alternative — attach to the ServiceAccount

If every pod in a namespace needs the same registry credential, set it once on the namespace's default ServiceAccount and drop the per-pod field:

```bash
kubectl patch serviceaccount default \
  -p '{"imagePullSecrets": [{"name": "regcred"}]}'
```

Now every pod in that namespace that doesn't override its ServiceAccount picks up `regcred` automatically. (ServiceAccounts land properly in notebook 10.) On our map this is the one section that reaches *up* out of the Config box to the **Registry** — specifically the **imagePullSecret** chip — because pulling a private image is where config meets the registry.